# Chapter 5: Importing Real-World Data and Initial Diagnostics

Companion notebook for *Master Data Cleaning: A Hands-On Beginner's Guide with Python, pandas & NumPy*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 5.1 Reading CSV Files with pd.read_csv()

### `pd.read_csv()`

In [1]:
import numpy as np
import pandas as pd

sales = pd.read_csv('sales_data.csv')
sales

,Date,Product,Units_Sold,Revenue,Region
0,2024-01-01,Widget A,10,250,North
1,2024-01-02,Widget B,5,150,South
2,2024-01-03,Widget A,8,200,North
3,2024-01-04,Widget C,15,450,West
4,2024-01-05,Widget B,12,360,East


In [2]:
from pathlib import Path

Path('sales_data.csv').exists()

True

### Files with other separators

In [3]:
pd.read_csv('sales_semicolon.csv', sep=';')

,Date,Product,Units_Sold,Revenue,Region
0,2024-01-01,Widget A,10,250,North
1,2024-01-02,Widget B,5,150,South
2,2024-01-03,Widget A,8,200,North


### Files without a header row

In [4]:
pd.read_csv(
    'sales_noheader.csv',
    header=None,
    names=['Date', 'Product', 'Units', 'Revenue', 'Region'],
)

,Date,Product,Units,Revenue,Region
0,2024-01-01,Widget A,10,250,North
1,2024-01-02,Widget B,5,150,South
2,2024-01-03,Widget A,8,200,North


### Title lines above the header

In [5]:
pd.read_csv('sales_report.csv', header=1)

,Date,Product,Units_Sold,Revenue,Region
0,2024-01-01,Widget A,10,250,North
1,2024-01-02,Widget B,5,150,South
2,2024-01-03,Widget A,8,200,North


### Setting the index on import

In [6]:
pd.read_csv('sales_data.csv', index_col='Date')

,Product,Units_Sold,Revenue,Region
Date,,,,
2024-01-01,Widget A,10,250,North
2024-01-02,Widget B,5,150,South
2024-01-03,Widget A,8,200,North
2024-01-04,Widget C,15,450,West
2024-01-05,Widget B,12,360,East


In [7]:
pd.read_csv('sales_data.csv', index_col=['Region', 'Product'])

,,Date,Units_Sold,Revenue
Region,Product,,,
North,Widget A,2024-01-01,10,250
South,Widget B,2024-01-02,5,150
North,Widget A,2024-01-03,8,200
West,Widget C,2024-01-04,15,450
East,Widget B,2024-01-05,12,360


### Parsing dates

In [8]:
sales.dtypes

Date            str
Product         str
Units_Sold    int64
Revenue       int64
Region          str
dtype: object

In [9]:
sales = pd.read_csv('sales_data.csv', parse_dates=['Date'])
sales.dtypes

Date          datetime64[us]
Product                  str
Units_Sold             int64
Revenue                int64
Region                   str
dtype: object

In [10]:
daily_sales = pd.read_csv(
    'sales_data.csv', index_col='Date', parse_dates=True
)
daily_sales.index

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05'],
              dtype='datetime64[us]', name='Date', freq=None)

### Loading only part of a file

In [11]:
pd.read_csv('sales_data.csv', usecols=['Date', 'Revenue'], nrows=3)

,Date,Revenue
0,2024-01-01,250
1,2024-01-02,150
2,2024-01-03,200


### Custom missing-value markers with `na_values`

In [12]:
pd.read_csv('stock_counts.csv', na_values=['missing', '-'])

,Product,Store,Stock
0,Widget A,New York,40.0
1,Widget B,Philadelphia,NaN
2,Widget C,Los Angeles,NaN
3,Widget A,Denver,25.0


### Forcing column types with `dtype`

In [13]:
pd.read_csv('customers.csv')

,Customer_ID,Name,Postcode
0,42,Aisha Rahman,2134
1,107,Daniel Lim,10001
2,215,Priya Nair,7030


In [14]:
customers = pd.read_csv(
    'customers.csv', dtype={'Customer_ID': str, 'Postcode': str}
)
customers

,Customer_ID,Name,Postcode
0,0042,Aisha Rahman,02134
1,0107,Daniel Lim,10001
2,0215,Priya Nair,07030


### Encoding errors

In [15]:
pd.read_csv('suppliers.csv', encoding='latin-1')

,Supplier,City
0,Café Bonheur,Paris
1,Müller GmbH,Munich


### Reading large files in chunks

In [16]:
total_revenue = 0
row_count = 0

for chunk in pd.read_csv('sales_data.csv', chunksize=2):
    total_revenue += chunk['Revenue'].sum()
    row_count += len(chunk)

print(row_count, total_revenue)

5 1410


## 5.2 Reading Excel Files with pd.read_excel()

### `pd.read_excel()`

In [17]:
q1_sales = pd.read_excel('quarterly_sales.xlsx')
q1_sales

,Date,Product,Units,Revenue,Region
0,2024-01-15,Widget A,120,3600,North
1,2024-02-20,Widget B,85,2550,South
2,2024-03-10,Widget A,200,6000,East


In [18]:
q1_sales.dtypes

Date       datetime64[us]
Product               str
Units               int64
Revenue             int64
Region                str
dtype: object

### Choosing a sheet

In [19]:
pd.read_excel('quarterly_sales.xlsx', sheet_name='Q2_Sales')

,Date,Product,Units,Revenue,Region
0,2024-04-05,Widget C,60,1800,West
1,2024-05-18,Widget A,140,4200,North
2,2024-06-22,Widget B,95,2850,East


The next cell raises an error on purpose.

In [20]:
pd.read_excel('quarterly_sales.xlsx', sheet_name='Q5_Sales')

ValueError: Worksheet named 'Q5_Sales' not found

In [21]:
pd.ExcelFile('quarterly_sales.xlsx').sheet_names

['Q1_Sales', 'Q2_Sales', 'Q3_Sales', 'Q4_Sales']

### Reading several sheets at once

In [22]:
all_quarters = pd.read_excel('quarterly_sales.xlsx', sheet_name=None)

for sheet, data in all_quarters.items():
    print(sheet, data.shape, data['Revenue'].sum())

Q1_Sales (3, 5) 12150
Q2_Sales (3, 5) 8850
Q3_Sales (3, 5) 10350
Q4_Sales (3, 5) 12000


### Title rows and selected columns

In [23]:
pd.read_excel(
    'regional_report.xlsx', header=3, usecols='A,C:D', nrows=2
)

,Region,Target,Actual
0,North,10000,11250
1,South,9000,8400


### Excel pitfalls

## 5.3 Exporting Data with to_csv() and to_excel()

In [24]:
employees = pd.DataFrame({
    'Employee': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'Department': ['Sales', 'IT', 'Sales', 'HR'],
    'Salary': [75000, 85000, 65000, 70000],
    'Performance': [4.25, 3.8, np.nan, 4.0],
})
employees

,Employee,Department,Salary,Performance
0,Alice,Sales,75000,4.25
1,Bob,IT,85000,3.80
2,Charlie,Sales,65000,NaN
3,Diana,HR,70000,4.00


### `df.to_csv()`

In [25]:
employees.to_csv('employee_data.csv')
print(Path('employee_data.csv').read_text())

,Employee,Department,Salary,Performance
0,Alice,Sales,75000,4.25
1,Bob,IT,85000,3.8
2,Charlie,Sales,65000,
3,Diana,HR,70000,4.0



In [26]:
employees.to_csv('employee_data.csv', index=False)
print(Path('employee_data.csv').read_text())

Employee,Department,Salary,Performance
Alice,Sales,75000,4.25
Bob,IT,85000,3.8
Charlie,Sales,65000,
Diana,HR,70000,4.0



### Customizing the CSV output

In [27]:
employees.to_csv(
    'performance_report.csv',
    columns=['Employee', 'Performance'],
    index=False,
    na_rep='N/A',
    float_format='%.2f',
)
print(Path('performance_report.csv').read_text())

Employee,Performance
Alice,4.25
Bob,3.80
Charlie,N/A
Diana,4.00



### Appending to an existing file

In [28]:
new_hire = pd.DataFrame({
    'Employee': ['Eve'],
    'Department': ['Finance'],
    'Salary': [90000],
    'Performance': [4.3],
})
new_hire.to_csv(
    'employee_data.csv', mode='a', header=False, index=False
)
print(Path('employee_data.csv').read_text())

Employee,Department,Salary,Performance
Alice,Sales,75000,4.25
Bob,IT,85000,3.8
Charlie,Sales,65000,
Diana,HR,70000,4.0
Eve,Finance,90000,4.3



### `df.to_excel()`

In [29]:
employees.to_excel(
    'employee_data.xlsx', sheet_name='Staff', index=False
)
pd.read_excel('employee_data.xlsx', sheet_name='Staff')

,Employee,Department,Salary,Performance
0,Alice,Sales,75000,4.25
1,Bob,IT,85000,3.80
2,Charlie,Sales,65000,NaN
3,Diana,HR,70000,4.00


### Several sheets in one workbook with `pd.ExcelWriter()`

In [30]:
product_sales = pd.DataFrame({
    'Product': ['Widget A', 'Widget B'],
    'Q1_Sales': [120, 85],
    'Q2_Sales': [145, 90],
})

with pd.ExcelWriter('company_data.xlsx') as writer:
    product_sales.to_excel(writer, sheet_name='Sales', index=False)
    employees.to_excel(writer, sheet_name='Staff', index=False)

pd.ExcelFile('company_data.xlsx').sheet_names

['Sales', 'Staff']

In [31]:
quarterly_revenue = pd.DataFrame({
    'Quarter': ['Q1', 'Q2'],
    'Revenue': [250000, 280000],
})

with pd.ExcelWriter(
    'company_data.xlsx',
    engine='openpyxl',
    mode='a',
    if_sheet_exists='replace',
) as writer:
    quarterly_revenue.to_excel(writer, sheet_name='Finance',
                               index=False)

pd.ExcelFile('company_data.xlsx').sheet_names

['Sales', 'Staff', 'Finance']

### Choosing between CSV and Excel

### Export mistakes to avoid

The next cell raises an error on purpose.

In [32]:
employees.to_csv('reports/employees.csv', index=False)

OSError: Cannot save file into a non-existent directory: 'reports'

In [33]:
Path('output').mkdir(exist_ok=True)
employees.to_csv('output/employees.csv', index=False)

## 5.4 Other Common File Formats

### `pd.read_json()`

In [34]:
products = pd.read_json('products.json')
products

,sku,name,price,in_stock
0,W-A,Widget A,24.9,True
1,W-B,Widget B,29.9,False
2,W-C,Widget C,29.9,True


### `pd.read_sql()`

In [35]:
import sqlite3

connection = sqlite3.connect(':memory:')
employees.to_sql('employees', connection, index=False)

pd.read_sql(
    'SELECT Employee, Salary FROM employees WHERE Salary > 70000',
    connection,
)

,Employee,Salary
0,Alice,75000
1,Bob,85000


### `pd.read_parquet()`

In [36]:
sales.to_parquet('sales.parquet', index=False)
pd.read_parquet('sales.parquet').dtypes

Date          datetime64[us]
Product                  str
Units_Sold             int64
Revenue                int64
Region                   str
dtype: object

## 5.5 Numerical Diagnostics with describe()

In [37]:
staff = pd.DataFrame({
    'Age': [25, 28, 30, 45, 99],
    'Income': [60000, 75000, 40000, 120000, np.nan],
    'Department': ['Sales', 'HR', 'IT', 'Sales', 'Sales'],
    'Rating': [4.0, 3.8, 3.1, 4.5, 4.2],
})

### `df.describe()`

In [38]:
staff.describe()

,Age,Income,Rating
count,5.000000,4.000000,5.000000
mean,45.400000,73750.000000,3.920000
std,30.940265,34003.676272,0.526308
min,25.000000,40000.000000,3.100000
25%,28.000000,55000.000000,3.800000
50%,30.000000,67500.000000,4.000000
75%,45.000000,86250.000000,4.200000
max,99.000000,120000.000000,4.500000


### Choosing which columns to describe

In [39]:
staff.describe(include='all')

,Age,Income,Department,Rating
count,5.000000,4.000000,5,5.000000
unique,NaN,NaN,3,NaN
top,NaN,NaN,Sales,NaN
freq,NaN,NaN,3,NaN
mean,45.400000,73750.000000,NaN,3.920000
std,30.940265,34003.676272,NaN,0.526308
min,25.000000,40000.000000,NaN,3.100000
25%,28.000000,55000.000000,NaN,3.800000
50%,30.000000,67500.000000,NaN,4.000000
75%,45.000000,86250.000000,NaN,4.200000


In [40]:
staff.describe(include='str')

,Department
count,5
unique,3
top,Sales
freq,3


In [41]:
staff['Income'].describe()

count         4.000000
mean      73750.000000
std       34003.676272
min       40000.000000
25%       55000.000000
50%       67500.000000
75%       86250.000000
max      120000.000000
Name: Income, dtype: float64

## 5.6 Categorical Frequencies with value_counts()

In [42]:
orders = pd.DataFrame({
    'Order_ID': range(1001, 1011),
    'Status': ['Complete', 'Pending', 'Complete', 'complete',
               'Failed', 'Complete', 'Pending', 'Complete',
               'pending', 'Complete'],
    'Region': ['North', 'South', 'North', 'East', 'West',
               'North', 'South', 'north', 'South', 'North'],
    'Priority': ['High', 'Medium', 'Low', 'High', 'Medium',
                 'Low', 'High', 'Medium', 'High', 'Low'],
})

### `s.value_counts()`

In [43]:
orders['Status'].value_counts()

Status
Complete    5
Pending     2
complete    1
Failed      1
pending     1
Name: count, dtype: int64

### Proportions with `normalize=True`

In [44]:
orders['Status'].value_counts(normalize=True)

Status
Complete    0.5
Pending     0.2
complete    0.1
Failed      0.1
pending     0.1
Name: proportion, dtype: float64

### Counting missing values with `dropna=False`

In [45]:
orders_missing = orders.copy()
orders_missing.loc[[2, 6], 'Status'] = np.nan

orders_missing['Status'].value_counts(dropna=False)

Status
Complete    4
NaN         2
Pending     1
complete    1
Failed      1
pending     1
Name: count, dtype: int64

### Sorting by label with `sort_index()`

In [46]:
orders['Priority'].value_counts().sort_index()

Priority
High      4
Low       3
Medium    3
Name: count, dtype: int64

### Combinations of columns with `df.value_counts()`

In [47]:
orders.value_counts(subset=['Region', 'Priority'])

Region  Priority
North   Low         3
South   High        2
North   High        1
South   Medium      1
East    High        1
West    Medium      1
north   Medium      1
Name: count, dtype: int64

### `s.unique()` and `s.nunique()`

In [48]:
orders['Status'].unique()

<ArrowStringArray>
['Complete', 'Pending', 'complete', 'Failed', 'pending']
Length: 5, dtype: str

In [49]:
orders['Status'].nunique()

5

## 5.7 Identifying Data Type Issues and Sentinel Values

### Numbers stored as text

In [50]:
survey = pd.read_csv('survey_data.csv')
survey.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   respondent_id  5 non-null      int64
 1   age            5 non-null      str  
 2   income         4 non-null      str  
 3   satisfaction   5 non-null      int64
dtypes: int64(2), str(2)
memory usage: 333.0 bytes


The next cell raises an error on purpose.

In [51]:
survey['age'].mean()

TypeError: Cannot perform reduction 'mean' with string dtype

In [52]:
age_numbers = pd.to_numeric(survey['age'], errors='coerce')
survey.loc[age_numbers.isna(), 'age']

2    unknown
Name: age, dtype: str

In [53]:
survey['age'] = pd.to_numeric(survey['age'], errors='coerce')
survey['income'] = pd.to_numeric(
    survey['income'].str.replace('$', '').str.replace(',', '')
)
survey.dtypes

respondent_id      int64
age              float64
income           float64
satisfaction       int64
dtype: object

### Sentinel values

In [54]:
survey.describe()

,respondent_id,age,income,satisfaction
count,5.000000,4.000000,4.000000,5.000000
mean,3.000000,33.500000,33500.250000,-196.600000
std,1.581139,6.454972,23699.022489,448.555794
min,1.000000,28.000000,-999.000000,-999.000000
25%,2.000000,28.750000,28250.250000,3.000000
50%,3.000000,32.000000,41500.000000,4.000000
75%,4.000000,36.750000,46750.000000,4.000000
max,5.000000,42.000000,52000.000000,5.000000


### `df.replace()`

In [55]:
survey = survey.replace(-999, np.nan)
survey

,respondent_id,age,income,satisfaction
0,1,28.0,45000.0,4.0
1,2,35.0,NaN,5.0
2,3,NaN,52000.0,3.0
3,4,42.0,38000.0,NaN
4,5,29.0,NaN,4.0


### Why whole numbers turn into decimals

In [56]:
survey['satisfaction'].dtype

dtype('float64')

### Fixing problems at import time

In [57]:
pd.read_csv('survey_data.csv', na_values=['unknown', -999])

,respondent_id,age,income,satisfaction
0,1,28.0,"$45,000",4.0
1,2,35.0,NaN,5.0
2,3,NaN,"$52,000",3.0
3,4,42.0,"$38,000",NaN
4,5,29.0,NaN,4.0
